# pretrained.py from dewml.dev

The cell from [Dew's landing page](https://dewml.dev/), as it is there. It needs more memory than the page's shared CPU hosts have, so it runs here, on a Colab GPU (Runtime → Change runtime type → a GPU).

In [ ]:
!uv pip install --system -q "dewml[cuda12,streaming,quantization]"

In [ ]:
import jax.numpy as jnp

from dew import Trainer
from dew.config import OptimConfig
from dew.data import load
from dew.interop import PretrainedDecoder
from dew.objectives.lm import LMObjective
from dew.sampling import Sampling

bundle = PretrainedDecoder.load("Qwen/Qwen3-0.6B", dtype=jnp.bfloat16, param_dtype=jnp.bfloat16,
                                max_seq_len=128)
task = bundle.text_generation(sampling=Sampling(temperature=0))
print(task("The capital of France is", 12, key=0).text[0])

data = load("hf/winglian/tiny-shakespeare", batch=1, tokenizer="Qwen/Qwen3-0.6B", seq_len=64)
trainer = Trainer(LMObjective(bundle, seq_len=64), OptimConfig(learning_rate=1e-5), key=0)
state = trainer.fit(data, steps=1)
bundle.save("export", variables=state.variables)